# Phase 4: clean

Check the parsed tables against the rules below, fix or drop what breaks them, and keep a **cleaning log** of how many things each rule found. Most counts are expected to be 0, because the dataset is already tidy; the point is that we checked.

## Rules we check

| # | Rule | Action |
|---|---|---|
| 1 | image fails to load or is not RGB | drop photo and its boxes |
| 2 | duplicate image_id | keep the first |
| 3 | XML size differs from the image | drop photo and its boxes |
| 4 | box outside the photo | clip to the photo |
| 5 | box with zero or negative size | drop |
| 6 | unknown code (typo) | drop |
| 7 | exact duplicate box | drop |
| 8 | box of a code we treat as 'none' (D43, D44, D50) | set aside, not needed for labelling squares |
| 9 | tiny box (shorter side < 8 px) | kept, flagged in the 'tiny' column |

In [1]:
import sys
from pathlib import Path

import pandas as pd
from PIL import Image

sys.path.insert(0, str(Path.cwd().parents[1]))
from ml import config

src = config.PROCESSED / "03_parse"
if not (src / "images.csv").exists():  # the Phase 3 files may still be directly in processed/
    src = config.PROCESSED
images, boxes = pd.read_csv(src / "images.csv"), pd.read_csv(src / "boxes.csv")

MIN_SIDE = 8  # a box with a shorter side than this (px) is treated as noise
log = []


def record(rule, found, action):
    log.append({"rule": rule, "found": int(found), "action": action})
    print(f"{rule}: {int(found)}  ->  {action}")


print(len(images), "photos,", len(boxes), "boxes")

6000 photos, 8114 boxes


## 1. Photos

In [2]:
# 1. image files that cannot be opened, or are not RGB
broken = []
for r in images.itertuples():
    try:
        with Image.open(config.RAW / r.path) as im:
            im.load()
            if im.mode != "RGB":
                broken.append(r.image_id)
    except Exception:
        broken.append(r.image_id)
record("image fails to load or is not RGB", len(broken), "drop photo and its boxes")

# 2. duplicate image ids
record("duplicate image_id", images["image_id"].duplicated().sum(), "keep the first")

# 3. size in the XML differs from the real image
wrong = [r.image_id for r in images.itertuples() if Image.open(config.RAW / r.path).size != (r.width, r.height)]
record("XML size differs from the image", len(wrong), "drop photo and its boxes")

drop_images = set(broken) | set(wrong)
images = images[~images["image_id"].isin(drop_images)].drop_duplicates("image_id")
boxes = boxes[boxes["image_id"].isin(images["image_id"])]

image fails to load or is not RGB: 0  ->  drop photo and its boxes
duplicate image_id: 0  ->  keep the first


XML size differs from the image: 0  ->  drop photo and its boxes


## 2. Boxes

In [3]:
# 4. box coordinates outside the photo -> clip them to the photo
b = boxes.merge(images[["image_id", "width", "height"]], on="image_id")
clipped = b.assign(
    xmin=b.xmin.clip(0, b.width), xmax=b.xmax.clip(0, b.width),
    ymin=b.ymin.clip(0, b.height), ymax=b.ymax.clip(0, b.height),
)
outside = (clipped[["xmin", "ymin", "xmax", "ymax"]].to_numpy() != b[["xmin", "ymin", "xmax", "ymax"]].to_numpy()).any(axis=1)
record("box outside the photo", outside.sum(), "clip to the photo")
boxes = clipped.drop(columns=["width", "height"])

# 5. box with zero or negative width/height (after clipping)
empty = (boxes.xmax <= boxes.xmin) | (boxes.ymax <= boxes.ymin)
record("box with zero or negative size", empty.sum(), "drop")
boxes = boxes[~empty]

# 6. unknown code (typo)
unknown = boxes["cls"] == "drop"
record("unknown code (typo)", unknown.sum(), "drop")
boxes = boxes[~unknown]

# 7. exact duplicate boxes
dup = boxes.duplicated(["image_id", "cls", "xmin", "ymin", "xmax", "ymax"])
record("exact duplicate box", dup.sum(), "drop")
boxes = boxes[~dup]

# 8. boxes that are not one of our 4 damage classes (D43 / D44 / D50 became "none")
other = boxes["cls"] == "none"
record("box of a code we treat as 'none' (D43, D44, D50)", other.sum(), "set aside, not needed for labelling squares")
boxes = boxes[~other]

# 9. tiny boxes: flag them, do not delete (Phase 6 decides after the resize)
boxes = boxes.assign(tiny=(boxes[["xmax", "xmin"]].diff(axis=1)["xmin"].abs().lt(MIN_SIDE)) | (boxes.ymax - boxes.ymin < MIN_SIDE))
record(f"tiny box (shorter side < {MIN_SIDE} px)", boxes["tiny"].sum(), "kept, flagged in the 'tiny' column")

box outside the photo: 0  ->  clip to the photo
box with zero or negative size: 0  ->  drop
unknown code (typo): 1  ->  drop
exact duplicate box: 0  ->  drop
box of a code we treat as 'none' (D43, D44, D50): 1885  ->  set aside, not needed for labelling squares
tiny box (shorter side < 8 px): 0  ->  kept, flagged in the 'tiny' column


## 3. Log and totals

In [4]:
log_table = pd.DataFrame(log)
log_table

,rule,found,action
0,image fails to load or is not RGB,0,drop photo and its boxes
1,duplicate image_id,0,keep the first
2,XML size differs from the image,0,drop photo and its boxes
3,box outside the photo,0,clip to the photo
4,box with zero or negative size,0,drop
5,unknown code (typo),1,drop
6,exact duplicate box,0,drop
7,"box of a code we treat as 'none' (D43, D44, D50)",1885,"set aside, not needed for labelling squares"
8,tiny box (shorter side < 8 px),0,"kept, flagged in the 'tiny' column"


In [5]:
print("photos:", len(pd.read_csv(src / "images.csv")), "->", len(images))
print("boxes: ", len(pd.read_csv(src / "boxes.csv")), "->", len(boxes))
print()
print(boxes["cls"].value_counts().rename("boxes").to_frame().T.to_string())

photos: 6000 -> 6000
boxes:  8114 -> 6228

cls     D00   D20   D40   D10
boxes  1951  1836  1359  1082


In [6]:
OUT = config.PROCESSED / "04_clean"
OUT.mkdir(parents=True, exist_ok=True)
images.to_csv(OUT / "images_clean.csv", index=False)
boxes.to_csv(OUT / "boxes_clean.csv", index=False)
log_table.to_csv(OUT / "cleaning_log.csv", index=False)
print("saved to", OUT)

saved to /Users/user/Documents/Y4/ML/road-damage-classifier/backend/data/processed/04_clean
